
# Planeación de trayectoria mediante Wavefront ponderado

## Resumen

En este trabajo se desarrolla un algoritmo de planeación de trayectoria para un robot móvil que se desplaza dentro de una cuadrícula bidimensional. El robot puede moverse hacia cualquiera de sus ocho celdas vecinas, de manera semejante al movimiento de un rey en el ajedrez. Los desplazamientos horizontales y verticales tienen un costo de $(1)$, mientras que los movimientos diagonales tienen un costo de $(1.4)$.

La metodología se basa en la transformación de distancia Wavefront. Debido a que existen diferentes costos de movimiento, la propagación se implementa mediante una cola de prioridad, siguiendo el principio del algoritmo de Dijkstra. De esta forma, cada celda libre almacena el costo mínimo necesario para alcanzar la meta.

También se incorpora una restricción para evitar que el robot atraviese diagonalmente las esquinas de los obstáculos. Esta condición permite representar de forma más realista las limitaciones físicas de un robot móvil.



## 1. Introducción

La planeación de trayectorias es una tarea fundamental en robótica móvil. Su propósito es encontrar una secuencia de posiciones que permita trasladar un robot desde una ubicación inicial hasta una meta, evitando colisiones con los obstáculos presentes en el entorno.

Cuando el espacio de trabajo se representa mediante una cuadrícula, cada celda puede clasificarse como libre u ocupada. Sobre esta representación se pueden aplicar algoritmos de búsqueda y transformación de distancia. El método Wavefront propaga valores desde la meta hacia las celdas libres del mapa y posteriormente obtiene la trayectoria siguiendo una secuencia de costos decrecientes.

En el documento de referencia, los movimientos ortogonales y diagonales pueden representarse mediante costos proporcionales a \(10\) y \(14\). En este trabajo se utilizan directamente los valores \(1\) y \(1.4\), conservando aproximadamente la relación geométrica entre una unidad de desplazamiento cartesiano y la distancia diagonal \(\sqrt{2}\).

La principal diferencia respecto a una transformación Wavefront uniforme es que no todos los movimientos tienen el mismo costo. Por esta razón, la expansión se realiza mediante una cola de prioridad que procesa primero las celdas con menor costo acumulado.



## 2. Objetivo

Desarrollar e implementar en Python un algoritmo de planeación de trayectoria que permita a un robot desplazarse en una cuadrícula utilizando ocho posibles movimientos, con un costo de \(1\) para los desplazamientos horizontales y verticales, y de \(1.4\) para los movimientos diagonales.

El algoritmo debe encontrar una trayectoria de costo mínimo desde la posición inicial hasta la meta, evitando los obstáculos y restringiendo los movimientos diagonales que atraviesen las esquinas de las celdas ocupadas.



## 3. Representación del entorno

El entorno se representa mediante una matriz de \(10\times10\) celdas. Cada elemento utiliza la codificación:

\[
M(i,j)=
\begin{cases}
0, & \text{si la celda está libre},\\
1, & \text{si la celda contiene un obstáculo}.
\end{cases}
\]

La posición inicial del robot se representa con la letra \(G\), mientras que la meta se representa con la letra \(R\). Para esta implementación se consideran las coordenadas \(G=(7,2)\) y \(R=(7,8)\), expresadas como \((\text{fila},\text{columna})\).


In [ ]:

import heapq
import numpy as np
import matplotlib.pyplot as plt

mapa = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
], dtype=int)

inicio = (7, 2)
meta = (7, 8)

print("Dimensiones del mapa:", mapa.shape)
print("Posición inicial:", inicio)
print("Posición meta:", meta)



## 4. Modelo de movimiento

El robot puede desplazarse hacia cualquiera de las ocho celdas vecinas. Los movimientos horizontales y verticales tienen un costo \(c=1\). Los movimientos diagonales tienen un costo \(c=1.4\), valor que aproxima \(\sqrt{2}\approx1.4142\).

El costo de una trayectoria completa se calcula mediante:

\[
C=\sum_{k=1}^{n} c_k
\]

donde \(c_k\) es el costo del movimiento realizado en el paso \(k\).


In [ ]:

movimientos = [
    (-1,  0, 1.0), ( 1,  0, 1.0),
    ( 0, -1, 1.0), ( 0,  1, 1.0),
    (-1, -1, 1.4), (-1,  1, 1.4),
    ( 1, -1, 1.4), ( 1,  1, 1.4)
]

for movimiento in movimientos:
    print(movimiento)



## 5. Validación de celdas y prevención de cruces por esquinas

Antes de aceptar un movimiento, se comprueba que la nueva posición permanezca dentro de los límites del mapa y que no corresponda a un obstáculo.

También se impide que el robot atraviese diagonalmente una esquina ocupada. Por ejemplo, para desplazarse desde \((i,j)\) hasta \((i+1,j+1)\), las celdas laterales \((i+1,j)\) y \((i,j+1)\) deben estar libres. Esta restricción evita el fenómeno conocido como *corner cutting*.


In [ ]:

def posicion_valida(mapa, fila, columna):
    """Comprueba que una celda esté dentro del mapa y se encuentre libre."""
    filas, columnas = mapa.shape
    if not (0 <= fila < filas and 0 <= columna < columnas):
        return False
    return mapa[fila, columna] == 0


def diagonal_permitida(mapa, actual, movimiento):
    """Impide atravesar diagonalmente las esquinas de los obstáculos."""
    fila, columna = actual
    df, dc, _ = movimiento

    if df == 0 or dc == 0:
        return True

    celda_vertical = (fila + df, columna)
    celda_horizontal = (fila, columna + dc)

    return (
        posicion_valida(mapa, *celda_vertical) and
        posicion_valida(mapa, *celda_horizontal)
    )



## 6. Transformación de distancia ponderada

La propagación comienza en la meta, a la cual se asigna un costo igual a cero:

\[
D(R)=0
\]

Para cada vecino \(v\) de una celda actual \(u\), se calcula:

\[
D_{\text{nuevo}}(v)=D(u)+c(u,v)
\]

Si este valor es menor que el costo almacenado previamente, se actualiza la celda. Debido a que existen costos diferentes, se emplea una cola de prioridad. En cada iteración se procesa la celda con menor costo acumulado, lo que corresponde al principio del algoritmo de Dijkstra.


In [ ]:

def calcular_mapa_costos(mapa, meta):
    """Calcula el costo mínimo desde cada celda libre hasta la meta."""
    filas, columnas = mapa.shape
    costos = np.full((filas, columnas), np.inf, dtype=float)
    costos[meta] = 0.0
    cola = [(0.0, meta[0], meta[1])]

    while cola:
        costo_actual, fila, columna = heapq.heappop(cola)

        if costo_actual > costos[fila, columna]:
            continue

        actual = (fila, columna)

        for movimiento in movimientos:
            df, dc, costo_movimiento = movimiento
            nf, nc = fila + df, columna + dc

            if not posicion_valida(mapa, nf, nc):
                continue
            if not diagonal_permitida(mapa, actual, movimiento):
                continue

            nuevo_costo = costo_actual + costo_movimiento

            if nuevo_costo < costos[nf, nc]:
                costos[nf, nc] = nuevo_costo
                heapq.heappush(cola, (nuevo_costo, nf, nc))

    return costos


costos = calcular_mapa_costos(mapa, meta)
np.set_printoptions(precision=1, suppress=True)
costos



## 7. Extracción de la trayectoria

Después de generar el mapa de costos, la trayectoria se obtiene desde la posición inicial. En cada paso se selecciona el vecino válido que tenga el menor costo:

\[
p_{k+1}=\underset{q\in N(p_k)}{\operatorname{argmin}}\;D(q)
\]

El procedimiento termina cuando se alcanza la meta. Si ninguna celda vecina posee un valor menor, se concluye que no existe una trayectoria válida desde la posición actual.


In [ ]:

def extraer_trayectoria(mapa, costos, inicio, meta):
    """Obtiene la ruta siguiendo valores decrecientes del mapa de costos."""
    if np.isinf(costos[inicio]):
        raise RuntimeError("La posición inicial no tiene conexión con la meta.")

    trayectoria = [inicio]
    actual = inicio

    while actual != meta:
        fila, columna = actual
        mejor_vecino = None
        mejor_costo = costos[fila, columna]

        for movimiento in movimientos:
            df, dc, _ = movimiento
            nf, nc = fila + df, columna + dc

            if not posicion_valida(mapa, nf, nc):
                continue
            if not diagonal_permitida(mapa, actual, movimiento):
                continue

            if costos[nf, nc] < mejor_costo:
                mejor_costo = costos[nf, nc]
                mejor_vecino = (nf, nc)

        if mejor_vecino is None:
            raise RuntimeError("No fue posible continuar hacia la meta.")

        actual = mejor_vecino
        trayectoria.append(actual)

    return trayectoria


trayectoria = extraer_trayectoria(mapa, costos, inicio, meta)
trayectoria



## 8. Verificación del costo

El costo almacenado en la posición inicial representa el costo mínimo hasta la meta. Para verificarlo, se realiza una suma independiente de todos los movimientos que forman la trayectoria.


In [ ]:

def calcular_costo_trayectoria(trayectoria):
    costo_total = 0.0
    cartesianos = 0
    diagonales = 0

    for actual, siguiente in zip(trayectoria[:-1], trayectoria[1:]):
        df = abs(siguiente[0] - actual[0])
        dc = abs(siguiente[1] - actual[1])

        if df == 1 and dc == 1:
            costo_total += 1.4
            diagonales += 1
        else:
            costo_total += 1.0
            cartesianos += 1

    return costo_total, cartesianos, diagonales


costo_total, n_cartesianos, n_diagonales = calcular_costo_trayectoria(trayectoria)

print("Trayectoria encontrada:")
for numero, posicion in enumerate(trayectoria):
    print(f"Paso {numero:2d}: {posicion}")

print("Número total de movimientos:", len(trayectoria) - 1)
print("Movimientos cartesianos:", n_cartesianos)
print("Movimientos diagonales:", n_diagonales)
print("Costo total calculado:", round(costo_total, 2))
print("Costo almacenado en el inicio:", round(costos[inicio], 2))



## 9. Visualización de resultados

La gráfica permite verificar que la trayectoria no atraviesa celdas ocupadas. Los valores dentro de las celdas libres corresponden al costo mínimo desde cada posición hasta la meta. Por esta razón, los valores disminuyen conforme la trayectoria se aproxima al objetivo.


In [ ]:

def mostrar_resultado(mapa, costos, trayectoria, inicio, meta):
    figura, eje = plt.subplots(figsize=(10, 10))
    eje.imshow(mapa, origin="upper", interpolation="none")

    for fila in range(mapa.shape[0]):
        for columna in range(mapa.shape[1]):
            if mapa[fila, columna] == 1:
                texto = "X"
            elif np.isinf(costos[fila, columna]):
                texto = "∞"
            else:
                texto = f"{costos[fila, columna]:.1f}"

            eje.text(columna, fila, texto, ha="center", va="center", fontsize=8)

    filas_ruta = [p[0] for p in trayectoria]
    columnas_ruta = [p[1] for p in trayectoria]

    eje.plot(columnas_ruta, filas_ruta, marker="o", linewidth=2, label="Trayectoria")
    eje.scatter(inicio[1], inicio[0], marker="s", s=180, label="Inicio G")
    eje.scatter(meta[1], meta[0], marker="*", s=250, label="Meta R")

    eje.set_xticks(np.arange(mapa.shape[1]))
    eje.set_yticks(np.arange(mapa.shape[0]))
    eje.set_xticks(np.arange(-0.5, mapa.shape[1], 1), minor=True)
    eje.set_yticks(np.arange(-0.5, mapa.shape[0], 1), minor=True)
    eje.grid(which="minor", linewidth=1)
    eje.tick_params(which="minor", bottom=False, left=False)

    eje.set_title("Planeación de trayectoria mediante Wavefront ponderado")
    eje.set_xlabel("Columna")
    eje.set_ylabel("Fila")
    eje.legend()
    plt.show()


mostrar_resultado(mapa, costos, trayectoria, inicio, meta)



## 10. Resultados y discusión

El algoritmo generó correctamente un mapa de costos desde la posición objetivo hacia todas las celdas libres alcanzables. Posteriormente, la ruta fue extraída desde la posición inicial mediante la selección sucesiva de vecinos con costos decrecientes.

La trayectoria obtenida rodea el obstáculo vertical y evita atravesar las esquinas ocupadas. El costo total calculado mediante la suma de movimientos coincide con el valor almacenado en la celda inicial del mapa de distancias. Esta coincidencia permite verificar la consistencia entre la fase de propagación y la fase de extracción.

La solución es óptima respecto al modelo discreto utilizado, porque la propagación se realiza mediante Dijkstra y todos los costos son positivos. Sin embargo, la trayectoria depende de la resolución de la cuadrícula, de la interpretación del mapa y de las restricciones impuestas al movimiento diagonal.

En una aplicación real también sería necesario considerar las dimensiones físicas del robot. Una forma habitual de hacerlo consiste en expandir los obstáculos una distancia equivalente al radio del robot antes de ejecutar el algoritmo.



## 11. Conclusiones

Se implementó un algoritmo de planeación de trayectoria capaz de trabajar con movimientos horizontales, verticales y diagonales. El uso de costos de \(1\) y \(1.4\) permite representar de manera aproximada la diferencia geométrica entre ambos tipos de desplazamiento.

La combinación de la transformación de distancia Wavefront con una cola de prioridad permite encontrar el costo mínimo desde cada celda hasta la meta. Posteriormente, la trayectoria puede recuperarse siguiendo los valores decrecientes del mapa de costos.

La restricción para evitar el cruce diagonal de esquinas mejora la validez física de la solución. El resultado demuestra que el robot puede trasladarse desde la posición inicial hasta la meta sin atravesar obstáculos y utilizando una trayectoria de costo mínimo bajo las reglas establecidas.

Como continuación del trabajo, el algoritmo puede adaptarse a mapas obtenidos mediante sensores, obstáculos dinámicos, CoppeliaSim, ROS 2 o un robot móvil real.



## 12. Referencia

McKerrow, P. J. *Introduction to Robotics*. Capítulo 8: Path Planning. Secciones relacionadas con la transformación de distancia, la generación de trayectorias y el suavizado de rutas.

El documento de referencia plantea la asignación de valores desde la meta hacia las celdas libres y la selección posterior de vecinos con valores decrecientes. En este notebook se adapta esa metodología a costos de \(1\) para movimientos cartesianos y \(1.4\) para movimientos diagonales.
